# CPI Forecasting — Random Forest

Predict **next month's CPI** from current economic indicators using a PySpark `RandomForestRegressor`, with a proper chronological train/test split and no data leakage.

In [ ]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-11.0.30.7-hotspot"

import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

spark = (
    SparkSession.builder
    .master("local[2]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.python.worker.reuse", "false")
    .appName("CPI_Prediction_RandomForest")
    .getOrCreate()
)

print("Spark started")

## Load and clean data

We restrict to a date window where **all features actually exist**, so we can simply drop NaN rows instead of imputing. This avoids the data-leakage problem that mean-imputation over the full dataset would cause.

In [ ]:
# ==========================================================================
# >>> CHANGE THESE TWO DATES TO EXPAND THE DATA WINDOW <<<
# Once more historical rows have all features populated, widen the range
# to give the model more training data.
# ==========================================================================
START_DATE = "2022-11-01"
END_DATE   = "2025-12-31"

file_path = "BigBoy.xlsx"
pdf = pd.read_excel(file_path)

pdf["Date"] = pd.to_datetime(pdf["Date"], errors="coerce")

numeric_cols = [
    "CPI", "Avg Predicted CPI", "Market Confidence Score",
    "Trading Intensity", "Fed Interest Rate (%)", "Unemployment Rate (%)",
    "Oil Prices (USD/bbl)", "Housing Prices", "S&P 500",
]

for c in numeric_cols:
    if c in pdf.columns:
        pdf[c] = pd.to_numeric(pdf[c], errors="coerce")

pdf = pdf.dropna(subset=["Date", "CPI"])
pdf = pdf[(pdf["Date"] >= START_DATE) & (pdf["Date"] <= END_DATE)]
pdf = pdf.sort_values("Date").reset_index(drop=True)

print(f"Rows after filtering to {START_DATE} – {END_DATE}: {len(pdf)}")
pdf.head(10)

## Create target variable and select features

The target is **next month's CPI** (`CPI_next`). We use `shift(-1)` so that each row's target is the CPI from the following month. The current CPI is included as a feature — it acts as a 1-month lag relative to the target.

In [ ]:
pdf["CPI_next"] = pdf["CPI"].shift(-1)

# Last row has no target — drop it
pdf = pdf.dropna(subset=["CPI_next"])

feature_cols = [
    "CPI",                        # current month CPI (lag-1 relative to target)
    "Avg Predicted CPI",
    "Market Confidence Score",
    "Trading Intensity",
    "Fed Interest Rate (%)",
    "Unemployment Rate (%)",
    "Oil Prices (USD/bbl)",
    "Housing Prices",
    "S&P 500",
]

# Drop rows where any feature or target is NaN — no imputation, no leakage
pdf = pdf.dropna(subset=feature_cols + ["CPI_next"])

pdf["Date_str"] = pdf["Date"].dt.strftime("%Y-%m-%d")

print(f"Rows available for modeling: {len(pdf)}")
print(f"Date range: {pdf['Date_str'].iloc[0]}  to  {pdf['Date_str'].iloc[-1]}")
pdf[["Date_str", "CPI", "CPI_next"]].head(10)

## Chronological train/test split

For time-series data we **must** split by date, not randomly. Otherwise the model can train on future data and test on the past, inflating metrics. We split on a specific date rather than using `limit()` + `subtract()`, which do not guarantee ordering in Spark.

In [ ]:
# ==========================================================================
# >>> ADJUST THIS SPLIT DATE WHEN CHANGING THE DATA WINDOW <<<
# Place it at roughly the 80 % mark of your date range.
# ==========================================================================
SPLIT_DATE = "2025-05-01"

spark_cols = ["Date_str"] + feature_cols + ["CPI_next"]
df = spark.createDataFrame(pdf[spark_cols])

train_df = df.filter(col("Date_str") < SPLIT_DATE)
test_df  = df.filter(col("Date_str") >= SPLIT_DATE)

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
train_df = assembler.transform(train_df)
test_df  = assembler.transform(test_df)

print(f"Train rows: {train_df.count()}  (before {SPLIT_DATE})")
print(f"Test rows:  {test_df.count()}  ({SPLIT_DATE} onwards)")

## Naive baseline

Predicts that next month's CPI equals this month's CPI. Any useful model should beat this.

In [ ]:
def evaluate(pred_df, label="CPI_next", pred="prediction"):
    metrics = {}
    for name in ["rmse", "mae", "r2"]:
        ev = RegressionEvaluator(labelCol=label, predictionCol=pred, metricName=name)
        metrics[name.upper()] = ev.evaluate(pred_df)
    return metrics

naive_predictions = test_df.withColumn("prediction", col("CPI"))

naive_metrics = evaluate(naive_predictions)
print("=== Naive baseline (CPI stays the same) ===")
for k, v in naive_metrics.items():
    print(f"  {k}: {v:.4f}")

## Random Forest model

In [ ]:
# ==========================================================================
# >>> TUNE THESE HYPERPARAMETERS ONCE YOU HAVE MORE DATA <<<
# With ~30 rows, keep maxDepth low to limit overfitting.
# numTrees can stay high — more trees don't overfit, they just average out.
# ==========================================================================
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="CPI_next",
    numTrees=100,
    maxDepth=4,
    minInstancesPerNode=2,
    seed=42,
)

rf_model = rf.fit(train_df)
rf_predictions = rf_model.transform(test_df)

rf_predictions.select("Date_str", "CPI", "CPI_next", "prediction").show(truncate=False)

rf_metrics = evaluate(rf_predictions)
print("\n=== Random Forest ===")
for k, v in rf_metrics.items():
    print(f"  {k}: {v:.4f}")

## Accuracy plots

In [ ]:
import matplotlib.pyplot as plt

plot_df = (
    rf_predictions
    .select("Date_str", "CPI_next", "prediction")
    .toPandas()
    .sort_values("Date_str")
)
plot_df["Date"] = pd.to_datetime(plot_df["Date_str"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Actual vs Predicted over time ---
ax = axes[0]
ax.plot(plot_df["Date"], plot_df["CPI_next"],    marker="o", label="Actual CPI (next month)")
ax.plot(plot_df["Date"], plot_df["prediction"],   marker="s", label="RF Prediction")
ax.set_xlabel("Date")
ax.set_ylabel("CPI")
ax.set_title("Actual vs Predicted CPI over Time")
ax.legend()
ax.tick_params(axis="x", rotation=45)

# --- Right: Scatter plot with ideal 45° line ---
ax = axes[1]
lo = min(plot_df["CPI_next"].min(), plot_df["prediction"].min()) - 0.5
hi = max(plot_df["CPI_next"].max(), plot_df["prediction"].max()) + 0.5
ax.plot([lo, hi], [lo, hi], "k--", alpha=0.5, label="Perfect prediction")
ax.scatter(plot_df["CPI_next"], plot_df["prediction"], edgecolors="steelblue", facecolors="none", s=60)
ax.set_xlabel("Actual CPI (next month)")
ax.set_ylabel("Predicted CPI")
ax.set_title("Prediction Accuracy (closer to line = better)")
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_aspect("equal")
ax.legend()

plt.tight_layout()
plt.show()

## Model comparison

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Naive baseline", "Random Forest"],
    "RMSE":  [naive_metrics["RMSE"], rf_metrics["RMSE"]],
    "MAE":   [naive_metrics["MAE"],  rf_metrics["MAE"]],
    "R2":    [naive_metrics["R2"],   rf_metrics["R2"]],
})

comparison

## Feature importance

Shows how much each feature contributed to the Random Forest's splits.

In [ ]:
importances = rf_model.featureImportances.toArray()

fi_df = (
    pd.DataFrame({"Feature": feature_cols, "Importance": importances})
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

fi_df

## Save predictions

In [ ]:
rf_predictions.select("Date_str", "CPI", "CPI_next", "prediction") \
    .toPandas() \
    .to_csv("cpi_predictions_random_forest.csv", index=False)

print("Saved to cpi_predictions_random_forest.csv")

## Save trained model

In [ ]:
model_path = "rf_cpi_model"
rf_model.write().overwrite().save(model_path)

print(f"Model saved to {model_path}/")